# Imports

In [133]:
import pandas as pd
import numpy as np
from joblib import Parallel, delayed
from tqdm import tqdm
import plotly.express as px

In [134]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 300)

In [135]:
file_path = 'E:/ML/datasets/mahjong/data/2019/block_5000.parquet'

In [136]:
states = pd.read_parquet(file_path)

In [137]:
states.columns = states.columns.map(
    str
)

In [138]:
dora_columns = []
for i in range(34,68):
    dora_columns = dora_columns + [str(i)]

In [139]:
hand_columns = []
for i in range(68,102):
    hand_columns = hand_columns + [str(i)]

In [140]:
states['round_id'] = states['511'].astype(str) + states['32'].astype(str)

In [141]:
columns_of_interest = ['round_id'] + ['0','1','2','5','6','7','8','9','10','11','12','13'] + dora_columns + hand_columns + ['510']

In [142]:
states = states[columns_of_interest] 

In [143]:
column_translations = {
    '0':'Round Wind',
    '1':'Dealer',
    '2':'POV Player',
    '5':'Wall Tiles',
    '6':'P0 Score',
    '7':'P1 Score',
    '8':'P2 Score',
    '9':'P3 Score',
    '10':'P0 Riichi',
    '11':'P1 Riichi',
    '12':'P2 Riichi',
    '13':'P3 Riichi',
    '510':'Discard'
}

In [144]:
states = states.rename(columns=column_translations)

In [145]:
states[states.columns[1:]] = states[states.columns[1:]].astype(int)

In [146]:
states_dora_sum = states.loc[:, dora_columns].sum(axis=1)

In [147]:
states = states.loc[states_dora_sum == 1].copy()

In [148]:
single_round = states.loc[states['round_id'] == '009379d90']

# Round 2 Vec

In [149]:
def get_perspective_position(metadata_vector):
    n = metadata_vector['POV Player']
    arr = np.asarray(metadata_vector['P0 Score':'P3 Score'])
    if np.all(arr == arr[0]):
        return 'tie'
    position = int(np.sum(arr < arr[n]))

    position_dict = {0:'1st',
                     1:'2nd',
                     2:'3rd',
                     3:'4th'}
    return position_dict[position]

In [150]:
def drop_nth(s, n):
    mask = np.ones(len(s), dtype=bool)
    mask[n] = False
    return s.iloc[mask]

In [151]:
def id_to_tile(id):
    return id %34

In [152]:
def normalize_seats(seats_vector):
    persp_seat = seats_vector.iloc[-1]
    persp_dict = {persp_seat: "P",
                  (persp_seat+1)%4: "R",
                  (persp_seat-1)%4: "L",
                  (persp_seat+2)%4: "A"}
    return seats_vector.replace(persp_dict)

In [153]:
mahjong_tiles = [
    # Manzu (Characters)
    "1m", "2m", "3m", "4m", "5m", "6m", "7m", "8m", "9m",
    
    # Souzu (Bamboos)
    "1s", "2s", "3s", "4s", "5s", "6s", "7s", "8s", "9s",
    
    # Pinzu (Circles)
    "1p", "2p", "3p", "4p", "5p", "6p", "7p", "8p", "9p",
    
    # Winds
    "East", "South", "West", "North",
    
    # Dragons (Colors)
    "White", "Green", "Red"
]

In [205]:
def round_2_vec(rnd):
    # Metadata section
    metadata_vector = rnd.iloc[-1].drop(hand_columns)
    
    lookup = np.array(['E','S','W','N'])
    round_wind = lookup[metadata_vector['Round Wind']]

    seat_wind = lookup[(metadata_vector['POV Player'] - metadata_vector['Dealer']) % 4]

    if (metadata_vector['POV Player'] == metadata_vector['Dealer']):
        is_dealer = 'dealer'
    else:
        is_dealer = 'nondealer'

    if metadata_vector['Wall Tiles'] < 20:
        wall = 'WU20'
    else:
        wall = 'WOoA20'

    relative_score = get_perspective_position(metadata_vector)

    if metadata_vector['P0 Riichi':'P3 Riichi'].iloc[metadata_vector['POV Player']]:
        did_riichi = 'did_riichi'
    else:
        did_riichi = 'no_riichi'
        
    other_riichis = f'{sum(drop_nth(metadata_vector['P0 Riichi':'P3 Riichi'],metadata_vector['POV Player']))}_OR'

    dora_series = metadata_vector[dora_columns]

    dora = id_to_tile(int(dora_series.loc[dora_series==1].index[0]))

    dora = f'D{mahjong_tiles[dora]}'

    # Discard Section
    discards = rnd['Discard']
    discard_vector = pd.Series(np.take(mahjong_tiles, discards.values), index=discards.index)
    discard_vector = normalize_seats(rnd['POV Player']) + discard_vector
    
    return np.concatenate(([round_wind], [seat_wind], [wall], [relative_score], [did_riichi], [other_riichis], [dora], discard_vector))

In [30]:
round_2_vec(single_round)

array(['E', 'W', 'WOoA20', 'tie', 'no_riichi', '0_OR', 'D5m', 'ASouth',
       'LSouth', 'P8p', 'RSouth', 'AWhite', 'L9s', 'PWhite', 'R1p', 'A9s',
       'LRed', 'PEast', 'REast', 'A7p', 'LWest', 'P1m', 'R8p', 'A2p',
       'L9m', 'P2m', 'A4p', 'L6s', 'PWest', 'R8s', 'A3p', 'L1m', 'P5p',
       'R7p', 'A5p', 'L1m', 'P7s', 'A6s', 'L3s', 'A4s', 'LWhite', 'P7p'],
      dtype=object)

In [154]:
round_2_vec(single_round)

array(['E', 'W', 'WOoA20', 'tie', 'no_riichi', '0_OR', 'D5m', 'ASouth',
       'LSouth', 'P8p', 'RSouth', 'AWhite', 'L9s', 'PWhite', 'R1p', 'A9s',
       'LRed', 'PEast', 'REast', 'A7p', 'LWest', 'P1m', 'R8p', 'A2p',
       'L9m', 'P2m', 'A4p', 'L6s', 'PWest', 'R8s', 'A3p', 'L1m', 'P5p',
       'R7p', 'A5p', 'L1m', 'P7s', 'A6s', 'L3s', 'A4s', 'LWhite', 'P7p'],
      dtype=object)

# Get all files

# Identify Chiitoi States

In [155]:
is_chiitoi = (states[hand_columns].astype(int) == 2).sum(axis=1) == 6

In [156]:
is_chiitoi.value_counts()

False    2333524
True        7313
Name: count, dtype: int64

In [157]:
states['is_chiitoi'] = is_chiitoi

In [178]:
states

,round_id,Round Wind,Dealer,POV Player,Wall Tiles,P0 Score,P1 Score,P2 Score,P3 Score,P0 Riichi,P1 Riichi,P2 Riichi,P3 Riichi,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,Discard,is_chiitoi
0,009379d90,0,0,0,69,25,25,25,25,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,2,0,0,0,1,0,1,0,1,0,1,0,0,1,0,0,0,1,0,0,1,0,3,28,False
1,009379d90,0,0,1,68,25,25,25,25,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,1,0,0,1,0,0,0,1,1,1,0,0,0,0,1,0,1,0,3,1,28,False
2,009379d90,0,0,2,67,25,25,25,25,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,2,2,0,0,0,0,0,1,1,0,0,0,1,0,1,0,0,1,0,1,0,0,0,1,0,0,25,False
3,009379d90,0,0,3,66,25,25,25,25,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,2,0,0,0,0,0,1,2,0,0,1,0,0,1,0,0,0,0,1,1,0,0,1,0,1,1,0,0,0,0,0,28,False
4,009379d90,0,0,0,65,25,25,25,25,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,2,0,0,1,1,0,1,0,1,0,1,0,0,1,0,0,0,0,0,0,1,0,3,31,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2412913,883fb84512,1,3,3,24,30,24,15,32,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,1,1,1,1,1,0,1,1,1,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,1,33,False
2412914,883fb84512,1,3,0,23,24,15,32,30,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,2,2,2,0,0,0,1,0,1,1,1,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,22,False
2412915,883fb84512,1,3,1,22,15,32,30,24,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,1,0,1,1,0,0,0,0,1,0,0,1,2,0,0,0,1,2,2,0,0,0,0,0,0,0,0,0,0,0,1,False
2412916,883fb84512,1,3,2,22,32,30,24,15,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,1,0,1,0,1,0,0,0,0,0,2,0,0,0,0,23,False


In [197]:
mask = states.groupby('round_id')['is_chiitoi'] \
         .transform('any')

result = states[mask]

In [198]:
# 1. find the index of the first True for each group
first_real = (
    result[result['is_chiitoi']]
    .groupby('round_id', sort=False)
    .apply(lambda g: g.index[0])
    .rename('first_idx')
)

# 2. map that back onto every row in df
result = result.reset_index()  # make the original index a column so we can compare to first_idx
result['first_idx'] = result['round_id'].map(first_real)

# 3. keep only rows whose original index is <= that group’s first_idx
df_truncated = (
    result[result['index'] <= result['first_idx']]
    .drop(['first_idx', 'index', 'is_chiitoi'], axis=1)
    .reset_index(drop=True)
)

C:\Users\Efith\AppData\Local\Temp\ipykernel_26932\350692415.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.index[0])


In [199]:
df_truncated

,round_id,Round Wind,Dealer,POV Player,Wall Tiles,P0 Score,P1 Score,P2 Score,P3 Score,P0 Riichi,P1 Riichi,P2 Riichi,P3 Riichi,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,Discard
0,4f5e8a986,0,2,2,69,22,21,24,31,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,2,0,0,0,1,1,0,0,1,0,0,1,0,1,1,0,0,0,1,0,1,0,0,1,1,29
1,4f5e8a986,0,2,3,68,21,24,31,22,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,2,1,0,0,2,0,1,2,1,0,0,0,0,0,0,0,1,0,1,0,1,1,0,29
2,4f5e8a986,0,2,0,67,24,31,22,21,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,1,0,0,0,0,0,0,1,2,0,1,0,1,1,0,1,0,1,0,0,0,1,0,0,0,1,29
3,4f5e8a986,0,2,1,66,31,22,21,24,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,2,1,0,2,2,0,0,0,0,0,0,0,0,0,1,0,0,1,1,0,1,0,0,0,0,1,28
4,4f5e8a986,0,2,2,65,22,21,24,31,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1,2,0,0,0,1,1,0,0,1,0,0,1,0,1,1,0,0,0,1,0,0,0,0,1,1,17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115260,883fb84510,1,2,3,51,33,24,15,29,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,2,1,2,0,0,0,2,1,2,2,1,0,0,0,0,0,0,0,0,0,0,6
115261,883fb84510,1,2,0,50,24,15,29,33,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,1,0,0,1,1,0,1,2,0,0,0,2,0,0,0,0,2,0,0,0,0,1,33
115262,883fb84510,1,2,1,49,15,29,33,24,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1,0,1,0,0,0,1,0,1,0,1,0,2,0,0,0,0,0,0,0,0,0,0,2,1,0,0,0,2,1,0,0,8
115263,883fb84510,1,2,2,49,29,33,24,15,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,29


In [208]:
chiitois = df_truncated.groupby('round_id').apply(round_2_vec, include_groups=False)

In [209]:
chiitois.iloc[5]

array(['E', 'N', 'WOoA20', 'tie', 'no_riichi', '0_OR', 'D9m', 'R1p',
       'AWest', 'LNorth', 'PWest', 'R1s', 'A9s', 'L9p', 'P9p', 'RNorth',
       'AEast', 'L8p', 'PWhite', 'RRed', 'ASouth', 'LSouth', 'P1s',
       'REast', 'A6m', 'LEast', 'P7m', 'R9m', 'A2p', 'LGreen', 'P3p',
       'RGreen', 'A1p', 'LRed', 'P9m', 'RWhite', 'AWest', 'L9s', 'P3m',
       'RNorth', 'AEast', 'L8s', 'PWhite', 'R4s', 'A1s', 'LGreen', 'P4s'],
      dtype=object)